# Next Priority Runs — Exp43 / VS r32

Outputs are Drive-only. This notebook does not update `PROGRESS.md` and does not push result CSVs to GitHub.

Preset order:
- `p0_exp43_s24`: Exp43 C0/C1 on S24, VS init r=32
- `p1_exp43_s01`: Exp43 C0/C1 on S01, VS init r=32
- `p2_vs_s28`: S28 r=32 SD LoRA VS generation
- `p3_vs_s02`: S02 r=32 SD LoRA VS generation


In [5]:
# [1] GPU check
import torch
assert torch.cuda.is_available(), 'GPU 없음: Runtime > Change runtime type > GPU'
print('torch', torch.__version__)
print('GPU  ', torch.cuda.get_device_name(0))


torch 2.11.0+cu128
GPU   NVIDIA L4


In [4]:
# [2] Mount Drive and clone/pull repo
from google.colab import drive
drive.mount('/content/drive')

import os, subprocess
REPO = 'https://github.com/heegyukim4043/test_diffusion_EEG_visualstim_image.git'
WORKDIR = '/content/vsvi_project'

if os.path.exists(WORKDIR) and os.path.exists(os.path.join(WORKDIR, '.git')):
    subprocess.run(['git', '-C', WORKDIR, 'pull', 'origin', 'main'], check=True)
else:
    subprocess.run(['rm', '-rf', WORKDIR], check=True)
    subprocess.run(['git', 'clone', REPO, WORKDIR], check=True)

os.chdir(WORKDIR)
subprocess.run(['git', 'log', '--oneline', '-5'], check=False)


Mounted at /content/drive


CompletedProcess(args=['git', 'log', '--oneline', '-5'], returncode=0)

In [ ]:
# [3] Select preset and launch background run
# Change PRESET one at a time. Do not run multiple presets simultaneously on one GPU.
PRESET = 'p1_exp43_s01' # 'p0_exp43_s24'  # p0_exp43_s24 | p1_exp43_s01 | p2_vs_s28 | p3_vs_s02

# Exp43 default uses VI/class=60 => 432 train trials. Set FULL_VI=True for all VI trials.
FULL_VI = False

cmd = f"python -u launch_next_priority_runs.py --preset {PRESET}"
if FULL_VI:
    cmd += ' --full_vi'

print(cmd)
!cd /content/vsvi_project && {cmd}


In [ ]:
# [4] Monitor active process and GPU
!pgrep -af 'train_exp43_vi_lora.py|train_vs_re_lora_gen.py' || true
!nvidia-smi
!ls -lt /content/drive/MyDrive/vsvi_data/logs | head -20


In [ ]:
# [5] Tail latest log for the selected preset
import glob, os
patterns = {
    'p0_exp43_s24': '/content/drive/MyDrive/vsvi_data/logs/exp43_s24_c0c1_r32_tok16_*.log',
    'p1_exp43_s01': '/content/drive/MyDrive/vsvi_data/logs/exp43_s01_c0c1_r32_tok16_*.log',
    'p2_vs_s28': '/content/drive/MyDrive/vsvi_data/logs/vs_s28_lora_r32_tok16_*.log',
    'p3_vs_s02': '/content/drive/MyDrive/vsvi_data/logs/vs_s02_lora_r32_tok16_*.log',
}
logs = sorted(glob.glob(patterns[PRESET]), key=os.path.getmtime, reverse=True)
assert logs, f'No log found for {PRESET}'
LOG = logs[0]
print('LOG=', LOG)
!tail -n 120 "{LOG}"


In [ ]:
# [6] Find Drive-only result CSVs
!echo '=== Exp43 VI results ==='
!find /content/drive/MyDrive/vsvi_data/checkpoints_exp43_vi_lora -name 'results_exp43_vi_lora.csv' -ls 2>/dev/null | tail -30
!echo '=== VS results ==='
!find /content/drive/MyDrive/vsvi_data/checkpoints_vsre_lora_gen -name 'results_lora_gen.csv' -ls 2>/dev/null | tail -30


In [ ]:
# 셀 1: colab_setup.py 내용을 직접 생성
!cd /content/vsvi_project && wget -q "https://raw.githubusercontent.com/heegyukim4043/test_diffusion_EEG_visualstim_image/main/colab_setup.py" -O colab_setup.py 2>/dev/null || echo "Not on GitHub yet"

Not on GitHub yet


In [ ]:
# Colab 파일 업로드
from google.colab import files
uploaded = files.upload()  # colab_setup.py 선택
!mv colab_setup.py /content/vsvi_project/

Saving colab_setup.py to colab_setup.py
Saving colab_setup_and_run.py to colab_setup_and_run.py


In [ ]:
%%bash
cd /content/vsvi_project
git add colab_setup.py
git commit -m "Add colab_setup.py for runtime recovery"
git push

Author identity unknown

*** Please tell me who you are.

Run

  git config --global user.email "you@example.com"
  git config --global user.name "Your Name"

to set your account's default identity.
Omit --global to set the identity only in this repository.

fatal: unable to auto-detect email address (got 'root@7aea7b2fbc8d.(none)')
fatal: could not read Username for 'https://github.com': No such device or address


CalledProcessError: Command 'b'cd /content/vsvi_project\ngit add colab_setup.py\ngit commit -m "Add colab_setup.py for runtime recovery"\ngit push\n'' returned non-zero exit status 128.

In [ ]:
%%bash
cd /content/vsvi_project && python colab_setup.py 2>&1

In [ ]:
%%bash
ls -la /content/vsvi_project/colab_setup.py 2>/dev/null || echo "파일 없음"
ls /content/colab_setup.py 2>/dev/null || echo "루트에도 없음"
find /content -name "colab_setup.py" 2>/dev/null || echo "어디에도 없음"

-rw-r--r-- 1 root root 0 Jul  4 21:52 /content/vsvi_project/colab_setup.py
루트에도 없음
/content/vsvi_project/colab_setup.py


In [ ]:
%%bash
# 1. symlink 생성
ln -sfn /content/drive/MyDrive/vsvi_data/preproc_vi_re /content/vsvi_project/preproc_vi_re

# 2. 확인
ls /content/vsvi_project/preproc_vi_re/preproc_subj_01_1.npz && echo "✅ VI data OK"
ls /content/vsvi_project/preproc_data_vi/images/01.png && echo "✅ Images OK"
ls /content/vsvi_project/checkpoints_vsre_dino/20260530_095045_ch32_merged_ep200_supcon/subj01_best.pt && echo "✅ SupCon OK"
ls /content/drive/MyDrive/vsvi_data/checkpoints_vsre_lora_gen/20260617_074245_lora_r32_ep100/subj01_lora_best.pt && echo "✅ VS LoRA OK"

/content/vsvi_project/preproc_vi_re/preproc_subj_01_1.npz
✅ VI data OK
/content/vsvi_project/preproc_data_vi/images/01.png
✅ Images OK
/content/vsvi_project/checkpoints_vsre_dino/20260530_095045_ch32_merged_ep200_supcon/subj01_best.pt
✅ SupCon OK
/content/drive/MyDrive/vsvi_data/checkpoints_vsre_lora_gen/20260617_074245_lora_r32_ep100/subj01_lora_best.pt
✅ VS LoRA OK


In [ ]:
%%bash
pip install -q "torchao>=0.16.0" 2>/dev/null || pip install -q "peft==0.11.1" 2>/dev/null || pip install -q --no-deps "peft==0.13.2"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 112.8 MB/s eta 0:00:00


In [ ]:
%%bash
cd /content/vsvi_project
mkdir -p /content/drive/MyDrive/vsvi_data/checkpoints_exp43_vi_lora

LOG=/content/drive/MyDrive/vsvi_data/logs/exp43_s01_c0c1_r32_tok16_full_$(date +%Y%m%d_%H%M%S).log

nohup python -u train_exp43_vi_lora.py \
  --subject_ids 1 \
  --conditions c0,c1 \
  --data_root /content/vsvi_project/preproc_vi_re \
  --img_root /content/vsvi_project/preproc_data_vi/images \
  --supcon_ckpt /content/vsvi_project/checkpoints_vsre_dino/20260530_095045_ch32_merged_ep200_supcon \
  --init_lora_ckpt /content/drive/MyDrive/vsvi_data/checkpoints_vsre_lora_gen/20260617_074245_lora_r32_ep100/subj01_lora_best.pt \
  --ckpt_root /content/drive/MyDrive/vsvi_data/checkpoints_exp43_vi_lora \
  --lora_r 32 \
  --n_eeg_tokens 16 \
  --epochs 100 \
  --batch_size 1 \
  --per_class_total 135 \
  --eval_n_samples 54 \
  --fp16 \
  > "$LOG" 2>&1 &

echo "PID=$!"
echo "LOG=$LOG"

PID=7824
LOG=/content/drive/MyDrive/vsvi_data/logs/exp43_s01_c0c1_r32_tok16_full_20260704_215639.log


In [ ]:
%%bash
tail -n 10 /content/drive/MyDrive/vsvi_data/logs/exp43_s01_c0c1_r32_tok16_full_20260704_215639.log

  Exp43 C0  S01
  [Encoder] Loaded and frozen: /content/vsvi_project/checkpoints_vsre_dino/20260530_095045_ch32_merged_ep200_supcon/subj01_best.pt
  Loading SD 1.5 UNet...
trainable params: 6,377,472 || all params: 865,898,436 || trainable%: 0.7365
  [Init] C0 scratch VI LoRA (no VS LoRA initialization)
/content/vsvi_project/train_exp43_vi_lora.py:392: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = GradScaler() if use_fp16 else None
    Ep    TrLoss
/content/vsvi_project/train_exp43_vi_lora.py:426: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(enabled=use_fp16):


In [ ]:
%%bash
LOG=$(ls -t /content/drive/MyDrive/vsvi_data/logs/exp43_s01_*.log 2>/dev/null | head -1)
grep -E "train=|per_class_total" "$LOG"

  [dataset:train] S01 samples=972 eff_sessions=9 per_class_total=135 class_counts={0: 108, 1: 108, 2: 108, 3: 108, 4: 108, 5: 108, 6: 108, 7: 108, 8: 108}
  [dataset:val] S01 samples=117 eff_sessions=9 per_class_total=135 class_counts={0: 13, 1: 13, 2: 13, 3: 13, 4: 13, 5: 13, 6: 13, 7: 13, 8: 13}
  [dataset:test] S01 samples=126 eff_sessions=9 per_class_total=135 class_counts={0: 14, 1: 14, 2: 14, 3: 14, 4: 14, 5: 14, 6: 14, 7: 14, 8: 14}
[INFO] S01 train=972 val=117 test=126


In [ ]:
%%bash
LOG=$(ls -t /content/drive/MyDrive/vsvi_data/logs/exp43_s01_*.log 2>/dev/null | head -1)
tail -n 5 "$LOG"

/content/vsvi_project/train_exp43_vi_lora.py:392: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = GradScaler() if use_fp16 else None
    Ep    TrLoss
/content/vsvi_project/train_exp43_vi_lora.py:426: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(enabled=use_fp16):


In [8]:
%%bash
# 최신 로그 확인
LOG=$(ls -t /content/drive/MyDrive/vsvi_data/logs/exp43_s01_*.log 2>/dev/null | head -1)
echo "$LOG"
tail -n 20 "$LOG"

/content/drive/MyDrive/vsvi_data/logs/exp43_s01_c0c1_r32_tok16_full_20260705_004058.log
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/torch/nn/modules/module.py", line 1790, in _call_impl
    return forward_call(*args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/diffusers/models/attention.py", line 986, in forward
    norm_hidden_states = self.norm1(hidden_states)
                         ^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/torch/nn/modules/module.py", line 1779, in _wrapped_call_impl
    return self._call_impl(*args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/torch/nn/modules/module.py", line 1790, in _call_impl
    return forward_call(*args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/torch/nn/modules/normalization.py", line 229, in fo

In [6]:
%%bash
LOG=$(ls -t /content/drive/MyDrive/vsvi_data/logs/exp43_s01_*.log 2>/dev/null | head -1)
grep -E "Done Exp43|Saved run summary|top1" "$LOG" | tail -10

In [7]:
%%bash
find /content/drive/MyDrive/vsvi_data/checkpoints_exp43_vi_lora -name "*summary.csv" -exec echo "---" \; -exec cat {} \;

---
sid,condition,best_ep,best_loss,top1,top3,top5,dominant,entropy,per_class_total,init_lora_ckpt,save_dir
24,c0,94,0.0025650510030269893,0.09259259259259259,0.37037037037037035,0.6296296296296297,0.3148148148148148,1.313171437844515,60,,/content/drive/MyDrive/vsvi_data/checkpoints_exp43_vi_lora/20260703_184527_exp43_vi_c0_s24_lora_r32_tok16_ep100_percls60
24,c1,95,0.0012983284090026886,0.1111111111111111,0.37037037037037035,0.6111111111111112,0.42592592592592593,1.1477879474379677,60,/content/drive/MyDrive/vsvi_data/checkpoints_vsre_lora_gen/20260629_094904_lora_r32_ep100/subj24_lora_best.pt,/content/drive/MyDrive/vsvi_data/checkpoints_exp43_vi_lora/20260703_184527_exp43_vi_c1_s24_lora_r32_tok16_ep100_percls60


In [9]:
%%bash
LOG=$(ls -t /content/drive/MyDrive/vsvi_data/logs/exp43_s01_*.log 2>/dev/null | head -1)
echo "$LOG"
grep -c "Ep" "$LOG" 2>/dev/null || echo "아직 epoch 시작 안 됨"
tail -n 5 "$LOG"

/content/drive/MyDrive/vsvi_data/logs/exp43_s01_c0c1_r32_tok16_full_20260705_004058.log
1
           ^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/torch/nn/functional.py", line 2935, in layer_norm
    return torch.layer_norm(
           ^^^^^^^^^^^^^^^^^
torch.OutOfMemoryError: CUDA out of memory. Tried to allocate 10.00 MiB. GPU 0 has a total capacity of 14.56 GiB of which 9.81 MiB is free. Process 3228 has 7.28 GiB memory in use. Including non-PyTorch memory, this process has 7.28 GiB memory in use. Of the allocated memory 7.13 GiB is allocated by PyTorch, and 7.78 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://docs.pytorch.org/docs/stable/notes/cuda.html#optimizing-memory-usage-with-pytorch-cuda-alloc-conf)
